In [1]:
# Current conda environment does not support tmap, set this up separately
# !conda create -n tmap python=3.9
# !pip install ipykernel pandas faerun mhfp tqdm rdkit
# !pip install numpy==1.26.4
# !pip install networkx
# !conda install -c tmap tmap -y

In [2]:
from tqdm import tqdm
import numpy as np
import tmap as tm
from rdkit.Chem import AllChem
from mhfp.encoder import MHFPEncoder
from faerun import Faerun
from matplotlib.colors import ListedColormap

In [3]:
import pandas as pd
import glob

# Define the path pattern for CSV files
csv_files = glob.glob("../data/Assays-pXC50/*.csv")

# List to hold individual DataFrames
dataframes = []

# Process each CSV file
for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    df["target"] = csv_file.split("/")[-1].split(".")[0]
    
    # Add 'active' and 'inactive' columns
    df['activity'] = (df['pXC50'] > 7).replace({True: 'active', False: 'inactive'})
    
    # Append to the list
    dataframes.append(df)

# Concatenate all DataFrames into a single DataFrame
all_data = pd.concat(dataframes, ignore_index=True)
all_data["target_activity"] = all_data["target"] + "_" + all_data["activity"]
all_data.sample(10)

,SMILES,pXC50,target,activity,target_activity
5612,O=C1C2CCCCC2C=NN1CCCCN1CCN(c2nsc3ccccc23)CC1,8.80000,D2R,active,D2R_active
1906,COc1ccc2c(c1)OCCN(C)CCc1ccccc1C2,5.80000,D3R,inactive,D3R_inactive
15498,CC(=O)Nc1ccc(SCCCN2CCN(c3ccccc3C)CC2)cc1,7.68000,_5HT2A,active,_5HT2A_active
7373,Cc1ccc2c(C3CCN(CCN4CCNC4=O)CC3)cn(-c3ccc(F)cc3...,6.57000,D2R,inactive,D2R_inactive
1691,Cc1ccc(OCCNCCCOc2ccc(F)cc2)cc1,6.63001,D3R,inactive,D3R_inactive
7040,COc1ccccc1N1CCN(Cc2cn3nc(Cl)ccc3n2)CC1,7.10237,D2R,active,D2R_active
6647,COc1cccc2c1CCCC2C(=O)NCCN1CCN(c2ccccn2)CC1,5.61000,D2R,inactive,D2R_inactive
7822,CCN1CCN(C)CC(NC(=O)c2cc(Br)c(N(C)C)nc2OC)C1,7.12494,D2R,active,D2R_active
11651,O=C(NCCCN1CCN(c2cccc(Cl)c2Cl)CC1)c1ccc2c(c1)Cc...,5.59000,D2R,inactive,D2R_inactive
13362,NC(N)=NC(=O)c1ccc2c(c1)Cc1ccccc1-2,6.00000,_5HT2A,inactive,_5HT2A_inactive


In [ ]:
# Cache fp results
fp_cache = dict()

CONDITIONS = {
    "alzheimer": ["AChE", "MAOB"],
    "schizophrenia": ["D2R", "_5HT2A"],
    "parkinson": ["D2R", "D3R"],
}

disease_colors = {
    "alzheimer": {
        "AChE": "#ff00ff",
        "MAOB": "#0000ff"
    },
    "schizophrenia": {
        "D2R": "#ffff00",
        "_5HT2A": "#ff0000"
    },
    "parkinson": {
        "D2R": "#00ff00",
        # "D3R": "#a4c2f4"
        "D3R": "#9900ff"
    }
}

enc = MHFPEncoder(1024)

def plot_tmap(lf, df, colors, disease):
    # Reference: https://github.com/reymond-group/tmap/blob/c74b718a86843292ab6aad91b99196b0133faac9/src/_tmap/layout.hh#L181
    cfg = tm.LayoutConfiguration()

    # The size of the nodes, which affects the magnitude of their repelling force.
    # Decreasing this value generally resolves overlaps in a very crowded tree.
    cfg.node_size = 1 / 26
    # Number of repeats of the per-level layout algorithm
    cfg.mmm_repeats = 2
    # Sets the number of repeats of the scaling.
    cfg.sl_extra_scaling_steps = 5
    # The number of nearest neighbors used to create the k-nearest neighbor graph
    cfg.k = 20
    # Defines the (relative) scale of the graph
    cfg.sl_scaling_type = tm.RelativeToAvgLength
    # Returns: The x and y coordinates of the vertices, the ids of the vertices spanning the edges, and information on the graph
    x, y, s, t, graph_properties = tm.layout_from_lsh_forest(lf, cfg)

    type_labels, type_data = Faerun.create_categories(df["target"])

    cmap = ListedColormap([colors[t[1]] for t in type_labels])

    f = Faerun(view="front", coords=False, clear_color='#ffffff')
    f.add_scatter(
        "np_atlas",
        {
            "x": x,
            "y": y,
            "c": [
                type_data,
            ],
            "labels": df["SMILES"],
        },
        shader="smoothCircle",
        point_scale=5.0,
        max_point_size=20,
        legend_labels=[type_labels],
        categorical=[True],
        colormap=[cmap],
        series_title=["Type",],
        has_legend=True,
    )
    f.add_tree("np_atlas_tree", {"from": s, "to": t}, point_helper="np_atlas")
    f.plot(template="smiles", notebook_height=0, file_name=disease)
    return graph_properties


def mol_to_fp(smiles):
    if smiles in fp_cache:
        return fp_cache[smiles]

    try:
        mol = AllChem.MolFromSmiles(smiles)
        fp = tm.VectorUint(enc.encode_mol(mol))
        fp_cache[smiles] = fp
        return fp
    except Exception as e:
        print(e)
        fp_cache[smiles] = None
        return None

disease_graphs = dict()
for disease, targets in CONDITIONS.items():
    lf = tm.LSHForest(
        d=1024,  # d = dimensionality of the MinHashe vectors to be added
        l=64     # l = number of prefix trees used when indexing data
    )
    fps = list()
    df = all_data[
        all_data["target"].isin(targets) &
        (all_data["activity"] == "active") 
    ] # .sample(50)
    smiles = df["SMILES"].tolist()

    for smiles in tqdm(smiles):
        fp = mol_to_fp(smiles)
        if fp is not None:
            fps.append(fp)
        else:
            print(smiles)

    lf.batch_add(fps)
    lf.index()
    colors = disease_colors[disease]
    graph_properties = plot_tmap(lf, df, colors, disease)
    disease_graphs[disease] = (graph_properties, df.reset_index(drop=True))

  0%|          | 0/1471 [00:00<?, ?it/s]

100%|██████████| 1471/1471 [00:05<00:00, 282.04it/s]


/home/arthurcerveira/CoMPO-GPT/scripts/alzheimer.html

100%|██████████| 4812/4812 [00:15<00:00, 309.10it/s]


/home/arthurcerveira/CoMPO-GPT/scripts/schizophrenia.html

100%|██████████| 4714/4714 [00:03<00:00, 1494.05it/s]


/home/arthurcerveira/CoMPO-GPT/scripts/parkinson.html

In [8]:
import networkx as nx

disease_nx_graphs = dict()

for disease, (graph_properties, df) in disease_graphs.items():
    G = nx.Graph()
    class_dict = {}

    for i, row in df.iterrows():
        G.add_node(i)
        class_dict[i] = row['target']

    for i, adj in enumerate(graph_properties.adjacency_list):
        for j in adj:
            G.add_edge(i, j[0], weight=j[1])

    node_classes = class_dict
    nx.set_node_attributes(G, node_classes, 'class')
    print(len(G.nodes), len(G.edges))
    disease_nx_graphs[disease] = G

disease_nx_graphs

1471 1470
4812 4811
4714 4713


{'alzheimer': <networkx.classes.graph.Graph at 0x7f7cc50d86a0>,
 'schizophrenia': <networkx.classes.graph.Graph at 0x7f7cc50d8100>,
 'parkinson': <networkx.classes.graph.Graph at 0x7f7cc4083e80>}

In [10]:
import networkx as nx
import numpy as np

# G is your MST; edges have weight "w"; node attribute "label" in {'A','B'}
for disease, G in disease_nx_graphs.items():
    labels = nx.get_node_attributes(G, 'class')

    # Cross-target edges
    E = list(G.edges())
    W = np.array([G[u][v]['weight'] for u,v in E])

    cross_edges = [(u, v) for u, v in E if labels[u] != labels[v]]
    cross_idx = [i for i, (u, v) in enumerate(E) if labels[u] != labels[v]]
    xtef = len(cross_edges) / len(E)

    # Conductance-like evaluation
    classes = list(set(labels.values()))
    assert len(classes) == 2, "Expected exactly two classes (two targets)."

    target_A = {n for n, c in labels.items() if c == classes[0]}
    target_B = set(G.nodes()) - target_A

    def cut_weight(A, B):
        return sum(G[u][v]['weight'] for u, v in G.edges() 
                if (u in A and v in B) or (u in B and v in A))

    def vol(A):
        return sum(G[u][v]['weight'] 
                for u, v in G.edges() if u in A or v in A)

    cut = cut_weight(target_A, target_B)
    phi = cut / min(vol(target_A), vol(target_B))  # conductance

    # Modularity
    from networkx.algorithms.community.quality import modularity
    Q = modularity(G, [target_A, target_B], weight='weight')

    # -- Print results
    print(f"Disease: {disease}")
    print(f"XTEF:          {xtef:.2f}")
    print(f"Conductance φ: {phi:.2f}")
    print(f"Modularity Q:  {Q:.2f}\n")

Disease: alzheimer
XTEF:          0.03
Conductance φ: 0.14
Modularity Q:  0.41

Disease: schizophrenia
XTEF:          0.20
Conductance φ: 0.32
Modularity Q:  0.32

Disease: parkinson
XTEF:          0.36
Conductance φ: 0.53
Modularity Q:  0.20

